# CODY-SAM3 — Phase 3 - External validation - v1 (MANUAL calibration)

The calibration patient set is chosen **manually** for each dataset, and the *train-once -> calibrate -> deploy* strategy is evaluated in exact continuity with **CODY-2**.

**Calibration objective (identical to cody-2 `calibrate_dataset2.py`)**:
```
score = Jaccard(Y_true, Y_pred)
        - 0.35 * fp_excess_norm           # penalises over-calling
        - 0.35 * fn_key_rate(Dystonia, Myoclonus, Chorea)   # protects key labels
```
jointly optimised over the 8 phenomenologies (coordinate descent), with threshold guard-rails (min: Tremor/Tics 0.35, Ballismus 0.30, Stereotypies 0.25, Athetosis 0.20; max: Dystonia 0.55, Myoclonus 0.50, Chorea 0.55) and the aggregation grid p70/p90/p95/max.

**Outputs** (per dataset): calibrated rules, Hamming/Jaccard + TP/TN/FP/FN for each view (held-out / all / calibration) and each definition (main present/absent & restricted, agreement levels >=3/5, >=4/5, 5/5), per-phenomenology confusion table.

**GPU**: not required if the window-level probabilities already exist (`inference_window_predictions.csv.gz`). Otherwise run `phase_2_external_inference.ipynb` first.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q openpyxl pandas numpy matplotlib
from pathlib import Path
import subprocess, sys

## 2. Configuration — edit the calibration sets here

In [ ]:
DRIVE     = Path('/content/drive/MyDrive/sam_3_infer')
RESULTS   = DRIVE / 'external_validation'          # contient dataset_N/reports/tables/inference_window_predictions.csv.gz
GT_XLSX   = DRIVE / 'dataset_inference.xlsx'
SCRIPT    = DRIVE / 'cody_sam3_pipeline' / 'calibrate_persite.py'

# >>> SETS DE CALIBRATION CHOISIS MANUELLEMENT (un par dataset) <<<
#  dataset_2 = cohorte pediatrique CODY-2 -> meme set que le papier pour comparabilite directe
CALIB = {
    'dataset_1': 'P28',                 # 2 patients seulement -> illustratif
    'dataset_2': 'P3,P5,P8,P10,P11',    # set a priori clinicien CODY-2
    'dataset_3': 'P1,P10,P11,P12,P13',  # to define/adjust
}
SHEETS = {'dataset_1':'dataset_1','dataset_2':'dataset_2','dataset_3':'dataset_3'}

# Definition utilisee POUR la calibration (le papier calibre sur main >=3/5)
CALIB_FAMILY = 'main'
CALIB_LEVEL  = 3

## 3. Calibration + evaluation per dataset

Calls `calibrate_persite.py` (CODY-2 objective). Each dataset produces its own `eval_persite/` folder with rules, metrics and confusion tables.

In [ ]:
for ds, calib in CALIB.items():
    win = RESULTS / ds / 'reports' / 'tables' / 'inference_window_predictions.csv.gz'
    if not win.exists():
        print(f'[skip] {ds}: {win} not found (run phase_2 first)'); continue
    out = RESULTS / ds / 'eval_persite'
    print('='*70); print(f'  {ds}  | calib = {calib}'); print('='*70)
    subprocess.run([sys.executable, str(SCRIPT),
        '--win_csv', str(win), '--gt_xlsx', str(GT_XLSX),
        '--sheet', SHEETS[ds], '--dataset_tag', ds,
        '--calib', calib, '--out_dir', str(out),
        '--calib_family', CALIB_FAMILY, '--calib_level', str(CALIB_LEVEL)], check=False)

## 4. Summary table (CODY-2 Table 3 style)

In [ ]:
import pandas as pd
frames=[]
for ds in CALIB:
    f = RESULTS / ds / 'eval_persite' / f'persite_{ds}__metrics.csv'
    if f.exists(): frames.append(pd.read_csv(f))
if frames:
    allm = pd.concat(frames, ignore_index=True)
    view = allm[allm.view.isin(['heldout','all'])]
    piv = view.pivot_table(index=['dataset','definition','agreement'],
                           columns='view', values=['Hamming','Jaccard','TP','TN','FP','FN'])
    pd.set_option('display.width',220)
    display(piv.round(3))
    allm.to_csv(RESULTS/'SUMMARY_persite_manual.csv', index=False)
    print('-> SUMMARY_persite_manual.csv')
else:
    print('No results: run cell 3 first.')

## 5. Per-phenomenology confusion (CODY-2 Table 4 style)

In [ ]:
frames=[]
for ds in CALIB:
    f = RESULTS / ds / 'eval_persite' / f'persite_{ds}__perphen_confusion.csv'
    if f.exists(): frames.append(pd.read_csv(f))
if frames:
    pf = pd.concat(frames, ignore_index=True)
    sub = pf[(pf.definition=='main')&(pf.agreement=='3of5')&(pf.view.isin(['heldout','all']))]
    display(sub.pivot_table(index=['dataset','phenomenology'], columns='view',
                            values=['tp','tn','fp','fn']).fillna(0).astype(int))